In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
df = pd.read_csv("../data/ai4i2020.csv")   # use "data/ai4i2020.csv" if needed

df = df.rename(columns={
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear",
    "Machine failure": "failure"
})
df.head()

,UDI,Product ID,Type,air_temp,process_temp,rpm,torque,tool_wear,failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
df = df.drop(columns=["UDI", "Product ID"])
df.columns

Index(['Type', 'air_temp', 'process_temp', 'rpm', 'torque', 'tool_wear',
       'failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'],
      dtype='str')

In [4]:
df["Type"] = df["Type"].map({"L": 0, "M": 1, "H": 2})
df["Type"].value_counts()

Type
0    6000
1    2997
2    1003
Name: count, dtype: int64

In [7]:
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["wear_torque"] = df["tool_wear"] * df["torque"]
df["power"] = df["torque"] * df["rpm"] * 2 * np.pi / 60


In [8]:
df[["temp_diff", "power", "wear_torque"]].describe()

,temp_diff,power,wear_torque
count,10000.000000,10000.000000,10000.000000
mean,10.000630,6279.744953,4314.664550
std,1.001094,1067.418295,2826.567692
min,7.600000,1148.440610,0.000000
25%,9.300000,5561.184484,1963.650000
50%,9.800000,6271.027344,4012.950000
75%,11.000000,7003.002724,6279.000000
max,12.100000,10469.923005,16497.000000


In [9]:
failure_types = df[["TWF", "HDF", "PWF", "OSF", "RNF"]]

In [10]:
X = df.drop(columns=["failure", "TWF", "HDF", "PWF", "OSF", "RNF"])
y = df["failure"]

print(X.columns.tolist())
print(X.shape, y.shape)

['Type', 'air_temp', 'process_temp', 'rpm', 'torque', 'tool_wear', 'temp_diff', 'power', 'wear_torque']
(10000, 9) (10000,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean() * 100, y_test.mean() * 100)

(8000, 9) (2000, 9)
3.3875 3.4000000000000004


In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

X_train_scaled.head()

,Type,air_temp,process_temp,rpm,torque,tool_wear,temp_diff,power,wear_torque
4058,0.755329,0.998914,0.604282,-0.460607,0.718305,-0.843997,-1.100842,0.852387,-0.621945
1221,0.755329,-1.505194,-1.153260,-0.775574,0.638456,0.382263,1.299658,0.480363,0.646140
6895,0.755329,0.498092,1.077466,-1.007654,0.558607,0.460870,0.599512,0.184041,0.689544
9863,-0.740002,-0.553633,-0.139294,-0.709265,1.626586,-0.372359,0.899575,1.898861,0.151247
8711,-0.740002,-1.455112,-1.018064,1.070019,-1.128202,-0.906882,1.399679,-1.001687,-1.016909


In [14]:
import os
os.makedirs("../data/processed", exist_ok=True)   # use "data/processed" if needed

X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# failure types for Module 6 (same rows as train and test)
failure_types.loc[X_train.index].to_csv("../data/processed/types_train.csv", index=False)
failure_types.loc[X_test.index].to_csv("../data/processed/types_test.csv", index=False)

joblib.dump(scaler, "../models/scaler.joblib")

['../models/scaler.joblib']